<a href="https://colab.research.google.com/github/Omar-RojasGBF/lis5693/blob/main/lab-8/Lab_8_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install bertopic datasets openai datamapplot

In [ ]:
import requests
import io
import pandas as pd

url = "https://raw.githubusercontent.com/Omar-RojasGBF/lis5693/refs/heads/main/lab-5/lens-export.csv"
response = requests.get(url)
response.raise_for_status()

df = pd.read_csv(io.StringIO(response.text))

abstracts = list(df["Abstract"])
titles = list(df["Title"])

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer('thenlper/gte-small')


processed_abstracts = [str(x) if not (isinstance(x, float) and np.isnan(x)) else "" for x in abstracts]

embeddings = embedding_model.encode(processed_abstracts, show_progress_bar=True)

In [ ]:
embeddings.shape

In [ ]:
from umap import UMAP

umap_model = UMAP(
    n_components=5, min_dist=0.0, metric='cosine', random_state=42
)
reduced_embeddings = umap_model.fit_transform(embeddings)

In [ ]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric='euclidean', cluster_selection_method='eom'
).fit(reduced_embeddings)
clusters = hdbscan_model.labels_

len(set(clusters))

In [ ]:
import numpy as np

cluster = 0
for index in np.where(clusters==cluster)[0][:3]:
    print(processed_abstracts[index][:300] + "... \n")

In [ ]:
import pandas as pd

reduced_embeddings = UMAP(
    n_components=2, min_dist=0.0, metric='cosine', random_state=42
).fit_transform(embeddings)

df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
df["title"] = titles
df["cluster"] = [str(c) for c in clusters]

clusters_df = df.loc[df.cluster != "-1", :]
outliers_df = df.loc[df.cluster == "-1", :]

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2, c="grey")
plt.scatter(
    clusters_df.x, clusters_df.y, c=clusters_df.cluster.astype(int),
    alpha=0.6, s=2, cmap='tab20b'
)
plt.axis('off')

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
).fit(processed_abstracts, embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(0)

In [ ]:
topic_model.find_topics("topic modeling")

In [ ]:
topic_model.get_topic(22)

In [ ]:
topic_model.topics_[titles.index('BERTopic: Neural topic modeling with a class-based TF-IDF procedure')]

In [ ]:
fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings,
    width=1200,
    hide_annotations=True
)

fig.update_layout(font=dict(size=16))

In [ ]:
topic_model.visualize_barchart()

topic_model.visualize_heatmap(n_clusters=2)

topic_model.visualize_hierarchy()

For Lab 5, my code did not want to run at all. However, I was able to get it to run this time without any isues. I noticed a lot more words like "the" and "and" appearing this time than in Lab 5. This is most likely due to I cleaned my data in Lab 5 to prevent stop words and punctuation from appearing in ym results. I also feel like this lab's results were a smaller scale than lab 5's results. Since are clustering out text data, it might be natural for the final result to be smaller than the results of sentiment analysis. Trying to categorize opinions might feature a range of emotions/feelings than simply trying to cluster text data together.

Reflection 💭


---
I think my Google colab final works well again; I had major issues with running my code. I had to do some internet searching to know how to extract metadata from my csv file. I think I can see myself using large language models or embeddings right if libraries decide to integrate more AI models for search queries or recommondations.I recently attended a webinar over the integration of AI in libraries, and it really helped in thinking more creatively over how I can use what I learned in course in my future career.
